# Explore the Kafka mirror

This notebook reads data from the mirror broker: which topics it holds, the latest messages in a topic, what arrived recently, a live view of new records, and the consumer groups on the mirror. The work is done by the `neon_sage_edge` package in this repository; the notebook only calls it, so the same functions can also be used from scripts.

If your topics' values are written with a schema registry (Avro or JSON Schema), the package can deserialize them into readable records. Point it at the registry in the connection settings; without one, values are shown as text or binaries.

It only **reads**.

**Before you start**

1. Consult with system administrators to get access.
2. Either set `MIRROR_PASSWORD` in the environment before starting Jupyter, or type the password when the settings cell asks for it.

The notebook connects to the mirror's external listener, `localhost:9094` by default. If Jupyter runs on another machine, set `MIRROR_BOOTSTRAP` to `<mirror-host>:9094` and make sure the broker advertises that host.

In [1]:
# Installs the package (and its Avro / JSON Schema support) from this repository.
# If pip reports "externally-managed-environment", start Jupyter from a virtual environment instead.
%pip install --quiet -e "..[registry]"

Note: you may need to restart the kernel to use updated packages.


## Connection settings

Defaults match the mirror setup. Override any of them with environment variables: `MIRROR_BOOTSTRAP`, `MIRROR_USERNAME`, `MIRROR_PASSWORD`, and `MIRROR_SECURITY_PROTOCOL`.

**Schema registry and deserializers (optional).** Every message's key and value go through a deserializer, chosen with `VALUE_FORMAT` and `KEY_FORMAT`:

- `avro` or `json`: the schema registry's Avro or JSON Schema deserializer. It fetches the schema each record was written with, so older and newer schema versions both work.
- `string`: plain UTF-8 text, with no registry involved.

Set `SCHEMA_REGISTRY_URL` (no login is used) and values default to `avro`; keys default to `string`. If this machine can't resolve the registry's name, also set `SCHEMA_REGISTRY_IP` to the address to use, as `curl --resolve` does; that mapping applies inside this notebook only.

Instead of environment variables, you can also build the settings in the cell below, as the commented example shows.

In [ ]:
import os
from getpass import getpass

import pandas as pd
from neon_sage_edge import (KafkaSettings, MessageDecoder, MirrorReader, RegistrySettings, expand_json, format_row,
                            make_registry_client, per_minute_counts, registry_subjects, show_notes, time_range)

show_notes()  # print the package's notes (e.g. records it couldn't deserialize) as plain lines
pd.set_option("display.max_colwidth", 120)

kafka = KafkaSettings.from_env(prompt=getpass)
registry = RegistrySettings(url="",
                             ip="", 
                             value_format="avro",
                            )

registry_client = make_registry_client(registry)
reader = MirrorReader.connect(kafka, MessageDecoder.from_settings(registry, registry_client))

print("Mirror:", kafka.describe())
print("Schema registry:", registry.describe())

## 1. Topics on the mirror

Your mirrored topics, with partition counts and how many messages each holds.

In [ ]:
reader.topics()

,topic,partitions,messages
0,reading.sensor.csat3,1,0
1,reading.sensor.csat3b,1,5419273
2,reading.sensor.li7200_raw,1,5415405


## 2. Pick a topic

Set `TOPIC` to one of the topics above. The table shows, per partition, the oldest retained offset (`low`), the next offset to be written (`high`), and the difference.

In [ ]:
TOPIC = os.environ.get("NOTEBOOK_TOPIC", "reading.sensor.csat3b")
reader.watermarks(TOPIC)

,partition,low,high,messages
0,0,0,5411533,5411533


## 3. Latest messages

In [6]:
latest = reader.peek(TOPIC, n=20)
latest

,partition,offset,timestamp,key,value,headers
0,0,5411513,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38+00:00, 'ux_wind_speed': -1.91084301471...",None
1,0,5411514,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.050000+00:00, 'ux_wind_speed': -1.9185...",None
2,0,5411515,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.100000+00:00, 'ux_wind_speed': -1.7666...",None
3,0,5411516,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.150000+00:00, 'ux_wind_speed': -1.6817...",None
4,0,5411517,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.200000+00:00, 'ux_wind_speed': -1.5067...",None
5,0,5411518,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.250000+00:00, 'ux_wind_speed': -1.1209...",None
6,0,5411519,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.300000+00:00, 'ux_wind_speed': -1.1626...",None
7,0,5411520,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.350000+00:00, 'ux_wind_speed': -1.0622...",None
8,0,5411521,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.400000+00:00, 'ux_wind_speed': -1.1372...",None
9,0,5411522,2026-10-08 21:54:49.516000+00:00,59483,"{'source_id': '59483', 'site_id': 'CPER', 'readout_time': 2026-10-08 21:54:38.450000+00:00, 'ux_wind_speed': -1.1901...",None


If the values are JSON objects, or records from the schema registry, this spreads their fields into columns:

In [7]:
expand_json(latest)

,partition,offset,timestamp,key,headers,source_id,site_id,readout_time,ux_wind_speed,uy_wind_speed,uz_wind_speed,sonic_temperature,diagnostic,pitch,roll,board_temp,board_humidity
0,0,5411513,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38+00:00,-1.910843,3.572650,0.145576,27.875340,0,1.992443,1.809103,34.75,30.227989
1,0,5411514,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.050000+00:00,-1.918578,3.625636,0.153263,27.835720,0,1.992443,1.809103,34.75,30.227989
2,0,5411515,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.100000+00:00,-1.766601,3.864671,0.142124,27.895571,0,1.992443,1.809103,34.75,30.227989
3,0,5411516,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.150000+00:00,-1.681712,3.917535,0.083337,27.881290,0,1.992443,1.809103,34.75,30.227989
4,0,5411517,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.200000+00:00,-1.506754,4.123862,0.245290,27.911831,0,1.992443,1.809103,34.75,30.227989
5,0,5411518,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.250000+00:00,-1.120944,3.934501,0.350546,27.855801,0,1.992443,1.809103,34.75,30.227989
6,0,5411519,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.300000+00:00,-1.162655,3.787902,0.252876,27.849239,0,1.992443,1.809103,34.75,30.227989
7,0,5411520,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.350000+00:00,-1.062210,3.763354,0.306423,27.844820,0,1.992443,1.809103,34.75,30.227989
8,0,5411521,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.400000+00:00,-1.137221,3.728640,0.294248,27.854429,0,1.992443,1.809103,34.75,30.227989
9,0,5411522,2026-10-08 21:54:49.516000+00:00,59483,None,59483,CPER,2026-10-08 21:54:38.450000+00:00,-1.190101,3.935515,0.398367,27.854401,0,1.992443,1.809103,34.75,30.227989


## 4. What arrived recently

Messages from the last 15 minutes by record timestamp, and how many arrived each minute. MirrorMaker keeps the original timestamps, so these reflect when records were produced on the source.

In [8]:
recent = reader.read_since(TOPIC, minutes=15, max_messages=5_000)
print(f"{len(recent)} messages in the last 15 minutes")
per_minute_counts(recent)

Stopped at 5000 messages (the max_messages limit); pass a larger max_messages to read the rest.


5000 messages in the last 15 minutes


,messages
timestamp,
2026-10-08 21:40:00+00:00,830
2026-10-08 21:41:00+00:00,1220
2026-10-08 21:42:00+00:00,1230
2026-10-08 21:43:00+00:00,1220
2026-10-08 21:44:00+00:00,500


The earliest and latest `readout_time` in those messages, next to their Kafka timestamps. `readout_time` comes from inside each record (when the reading was taken), while the Kafka timestamp is when the record was produced, so comparing the two shows how long readings take to reach Kafka. Pass another field name to `time_range` to do the same for it.

In [9]:
time_range(recent, "readout_time")

,messages,min,max,span
field,,,,
readout_time,5000,2026-10-08 21:40:01+00:00,2026-10-08 21:44:10.950000+00:00,0 days 00:04:09.950000
timestamp (Kafka),5000,2026-10-08 21:40:32.787000+00:00,2026-10-08 21:44:37.146000+00:00,0 days 00:04:04.359000


## 5. Watch new messages arrive

Prints each new message for 30 seconds, and keeps them in `live`. Produce something on the source while this runs to see replication happen.

In [10]:
live = reader.watch(TOPIC, seconds=10, on_message=lambda row: print(format_row(row)))

partition 0  offset 5412343  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412344  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412345  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412346  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412347  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412348  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412349  key='59483'  value="{'source_id': '59483', 'site_id': 'CPER', 'readout_time': datetime.datetime(2026"
partition 0  offset 5412350  key='59483'  value="{'source_id': '59483', 'site_id': 

## 6. Schema registry

Only does something when a registry is configured. Lists the registry's subjects with their latest schema, which helps pick `VALUE_FORMAT`: an `AVRO` subject needs `avro`, a `JSON` one `json`. Protobuf isn't supported, because its deserializer needs the generated message classes.

In [12]:
registry_subjects(registry_client)

,subject,version,schema_id,type
0,aepg600m,1,108,AVRO
1,aepg600m_heated,1,194,AVRO
2,aquatroll200,1,134,AVRO
3,ce318n_metrics,1,120,AVRO
4,ce318n_raw,1,99,AVRO
...,...,...,...,...
95,tdr315n_metrics,2,208,AVRO
96,tdr315n_raw,2,207,AVRO
97,windmonitorhd,1,47,AVRO
98,windobserverii,1,107,AVRO
